In [2]:
!pip install groq


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from groq import Groq

GROQ_API_KEYS = [
]

GROQ_MODEL = "openai/gpt-oss-120b"   # check console.groq.com/docs/models for current free-tier options

groq_current_index = 0
groq_exhausted_today = set()

def build_groq_client(idx):
    return Groq(api_key=GROQ_API_KEYS[idx])

groq_client = build_groq_client(groq_current_index)

def rotate_groq_key():
    global groq_current_index, groq_client
    groq_exhausted_today.add(groq_current_index)
    for _ in range(len(GROQ_API_KEYS)):
        groq_current_index = (groq_current_index + 1) % len(GROQ_API_KEYS)
        if groq_current_index not in groq_exhausted_today:
            groq_client = build_groq_client(groq_current_index)
            print(f"  -> switched to Groq key #{groq_current_index + 1}/{len(GROQ_API_KEYS)}")
            return True
    return False

def is_groq_quota_error(e):
    msg = str(e).lower()
    return "rate_limit" in msg or "429" in msg or "quota" in msg

def call_groq_llm(songs_batch, system_prompt_text, response_schema_hint):
    """
    response_schema_hint: a plain-text description of the exact JSON
    shape you want (Groq's JSON mode is looser than Gemini's -- no
    strict schema enforcement, so spell it out clearly in the prompt).
    """
    user = build_user_prompt(songs_batch)  # reuse your existing prompt builder

    keys_tried = 0
    while keys_tried < len(GROQ_API_KEYS):
        for attempt in range(1, MAX_RETRIES_PER_KEY + 1):
            try:
                resp = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": system_prompt_text + "\n\n" + response_schema_hint},
        {"role": "user", "content": user + "\n\nIMPORTANT: Each song MUST have exactly 33 queries "
                                            "total -- exactly 3 queries per category, all 11 categories "
                                            "present, no exceptions. Count before responding."},
    ],
    response_format={"type": "json_object"},
    temperature=0.7,
    max_tokens=8000,   
)
                text = resp.choices[0].message.content.strip()
                parsed = json.loads(text)
                return parsed.get("results", parsed)
            except Exception as e:
                if is_groq_quota_error(e):
                    print(f"  Groq key #{groq_current_index + 1} hit limit -- rotating")
                    break
                else:
                    wait = RATE_LIMIT_SEC * attempt
                    print(f"  [Groq key #{groq_current_index + 1}, attempt {attempt}] "
                          f"error: {e} -- retrying in {wait:.0f}s")
                    time.sleep(wait)
        else:
            pass

        keys_tried += 1
        if not rotate_groq_key():
            print("ALL GROQ KEYS EXHAUSTED.")
            return "ALL_KEYS_EXHAUSTED"

    return None

In [25]:
import os
import json
import time
import random
import pandas as pd
from google import genai
from google.genai import types

SEMANTIC_SONGS_CSV = "data/semantic_songs.csv"      
TRAINING_QUERIES_CSV = "data/training_queries.csv" 
ROTATION_STATE_FILE = "data/rotation_state.json"  

API_KEYS = [
    "os.environ.get("GROQ_API_KEY", "")"
]
if os.environ.get("GEMINI_API_KEYS"):
    API_KEYS = [k.strip() for k in os.environ["GEMINI_API_KEYS"].split(",") if k.strip()]

GEMINI_MODEL = "llama-3.3-70b-versatile" 
BATCH_SIZE = 1                              
RATE_LIMIT_SEC = 4.0
MAX_RETRIES_PER_KEY = 2
RANDOM_SEED = 42

CATEGORIES = [
    "emotion",
    "activity",
    "artist",
    "similarity",
    "era",
    "occasion",
    "mood",
    "vocals",
    "language",
    "conversational",
    "casual_hinglish",
]
QUERIES_PER_CATEGORY = 3  
QUERY_COLUMNS = ["query_id", "song_id", "query","query_type", "category", "language"]

os.makedirs("data", exist_ok=True)
random.seed(RANDOM_SEED)
TARGET_SONG_IDS = [5643]

In [26]:
def load_rotation_state():
    if os.path.exists(ROTATION_STATE_FILE):
        with open(ROTATION_STATE_FILE) as f:
            return json.load(f)
    return {"current_index": 0, "exhausted_today": []}

def save_rotation_state(state):
    with open(ROTATION_STATE_FILE, "w") as f:
        json.dump(state, f)

rotation_state = load_rotation_state()
current_index = rotation_state["current_index"]
exhausted_today = set(rotation_state["exhausted_today"])

def build_client(idx):
    return genai.Client(api_key=API_KEYS[idx])

client = build_client(current_index)
print(f"Starting with API key #{current_index + 1}/{len(API_KEYS)}")
if exhausted_today:
    print(f"Keys already marked exhausted this run: {sorted(exhausted_today)}")

def rotate_key():
    global current_index, client
    exhausted_today.add(current_index)
    for _ in range(len(API_KEYS)):
        current_index = (current_index + 1) % len(API_KEYS)
        if current_index not in exhausted_today:
            client = build_client(current_index)
            rotation_state["current_index"] = current_index
            rotation_state["exhausted_today"] = list(exhausted_today)
            save_rotation_state(rotation_state)
            print(f"  -> switched to API key #{current_index + 1}/{len(API_KEYS)}")
            return True
    return False

def is_quota_error(e):
    msg = str(e)
    return "RESOURCE_EXHAUSTED" in msg or "429" in msg

Starting with API key #1/1


## prompting and llm calling fxns

In [27]:
SYSTEM_PROMPT = """You are generating training queries for an AI semantic music retrieval model.

Input:
- song name
- primary singer
- language
- semantic profile

The semantic profile is the ONLY source of truth for meaning.
Do not invent themes or emotions not supported by it.

Generate exactly {total} unique search queries:
{n_per_category} queries for each of the {n_categories} categories below.

Categories:
- emotion: feelings the listener wants to match
- activity: what the listener is doing
- theme: what the song is about
- artist: searches centered on the singer
- similarity: searches for songs like this one (ONLY this category may mention the song title)
- era: decade or generation based searches
- vocals: vocal style or singer characteristics
- language: language-specific searches
- situation: real-life situations where someone would play the song
- casual_hinglish: natural Hindi-English mixed searches used by Indian users
Return your answer as a JSON object of the exact shape:
{"results": [{"song_id": <int>, "queries": [{"query": "<string>", "category": "<string>"}]}]}
Rules:
1. Every query must express a DIFFERENT search intent.
2. Never generate paraphrases.
3. Write natural searches exactly how real users search.
4. Except for similarity queries, never mention the song title.
5. casual_hinglish must sound genuinely conversational, not translated English.
6. Keep queries short (3-10 words when possible).
7. Avoid generic searches like:
   - best songs
   - top music
   - good playlist
8. Use lowercase only.
9. Return ONLY valid JSON.
"""

RESPONSE_SCHEMA = {
    "type": "ARRAY",
    "items": {
        "type": "OBJECT",
        "properties": {
            "song_id": {"type": "INTEGER"},
            "queries": {
                "type": "ARRAY",
                "items": {
                    "type": "OBJECT",
                    "properties": {
                        "query": {"type": "STRING"},
                        "category": {"type": "STRING", "enum": CATEGORIES},
                    },
                    "required": ["query", "category"],
                },
            },
        },
        "required": ["song_id", "queries"],
    },
}


def build_user_prompt(songs_batch):
    lines = []
    for s in songs_batch:
        lines.append(
            f'- song_id={s["song_id"]}, name="{s["song_name"]}", '
            f'artist="{s["primary_singer"]}", language={s["language"]}\n'
            f'  semantic_profile: "{s["semantic_profile"]}"'
        )
    return "Songs:\n" + "\n".join(lines)


def call_llm(songs_batch):
    system = SYSTEM_PROMPT.format(
        n_categories=len(CATEGORIES),
        n_per_category=QUERIES_PER_CATEGORY,
        total=len(CATEGORIES) * QUERIES_PER_CATEGORY,
    )
    user = build_user_prompt(songs_batch)
    config = types.GenerateContentConfig(
        system_instruction=system,
        response_mime_type="application/json",
        response_schema=RESPONSE_SCHEMA,
        temperature=0.8,
    )

    keys_tried_this_call = 0
    while keys_tried_this_call < len(API_KEYS):
        for attempt in range(1, MAX_RETRIES_PER_KEY + 1):
            try:
                resp = client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=user,
                    config=config,
                )
                return json.loads(resp.text.strip())
            except Exception as e:
                if is_quota_error(e):
                    print(f"  key #{current_index + 1} hit quota -- rotating")
                    break
                else:
                    wait = RATE_LIMIT_SEC * attempt
                    print(f"  [key #{current_index + 1}, attempt {attempt}] "
                          f"non-quota error: {e} -- retrying in {wait:.0f}s")
                    time.sleep(wait)
        else:
            pass

        keys_tried_this_call += 1
        if not rotate_key():
            print("ALL API KEYS EXHAUSTED for today. Stopping cleanly -- "
                  "re-run this script tomorrow to resume.")
            return "ALL_KEYS_EXHAUSTED"

    return None

In [28]:
df_sem = pd.read_csv(SEMANTIC_SONGS_CSV)
df_sem["language"] = df_sem["language"].astype(str).str.strip().str.lower()

done_ids = set()
if os.path.exists(TRAINING_QUERIES_CSV):
    done_ids = set(pd.read_csv(TRAINING_QUERIES_CSV)["song_id"].tolist())
    print(f"Resuming: {len(done_ids)} songs already have queries, skipping them.")

df_todo = df_sem[~df_sem["song_id"].isin(done_ids)].reset_index(drop=True)
print(f"Targeting {len(df_todo)} songs out of {len(TARGET_SONG_IDS)} requested ids "
      f"(any not found simply aren't in semantic_songs.csv).")
 
missing = set(TARGET_SONG_IDS) - set(df_todo["song_id"])
if missing:
    print(f"NOTE: these ids have no semantic profile yet, skipping: {sorted(missing)}")

Resuming: 5901 songs already have queries, skipping them.
Targeting 1 songs out of 1 requested ids (any not found simply aren't in semantic_songs.csv).


C:\Users\Sumit\AppData\Local\Temp\ipykernel_15380\677067724.py:6: DtypeWarning: Columns (0: query_type) have mixed types. Specify dtype option on import or set low_memory=False.
  done_ids = set(pd.read_csv(TRAINING_QUERIES_CSV)["song_id"].tolist())


In [29]:
RESPONSE_SCHEMA_HINT = """
Return your answer as a JSON object of this exact shape, nothing else:
{"results": [{"song_id": <int>, "queries": [{"query": "<string>", "category": "<string>"}, ...]}, ...]}
"""

In [30]:
if os.path.exists(TRAINING_QUERIES_CSV):
    existing_q = pd.read_csv(TRAINING_QUERIES_CSV)
    before = len(existing_q)
    existing_q = existing_q[~existing_q["song_id"].isin(TARGET_SONG_IDS)]
    removed = before - len(existing_q)
    if removed:
        print(f"Removed {removed} pre-existing (partial) query rows for target ids before regenerating.")
    existing_q.to_csv(TRAINING_QUERIES_CSV, index=False, columns=QUERY_COLUMNS)
    qid_start = int(existing_q["query_id"].max()) + 1 if len(existing_q) else 1
else:
    qid_start = 1
 
qid = qid_start
 
# ------------------------------ MAIN LOOP ---------------------------- #
 
query_rows = []
t0 = time.time()
n_batches = (len(df_todo) + BATCH_SIZE - 1) // BATCH_SIZE
 
for b in range(n_batches):
    batch_df = df_todo.iloc[b * BATCH_SIZE: (b + 1) * BATCH_SIZE]
    songs_batch = batch_df.to_dict("records")
 
    result = call_groq_llm(songs_batch, SYSTEM_PROMPT, RESPONSE_SCHEMA_HINT)
 
    if result == "ALL_KEYS_EXHAUSTED":
        break
    if result is None:
        print(f"[batch {b}] failed on every key for non-quota reasons, skipping this batch.")
        continue
 
    for item in result:
        sid = item["song_id"]
        src_row = batch_df[batch_df["song_id"] == sid]
        if src_row.empty:
            continue
        src = src_row.iloc[0]
 
        for q in item.get("queries", []):
            query_rows.append({
                "query_id": qid,
                "song_id": sid,
                "query": q.get("query", ""),
                "query_type": "generated",
                "category": q.get("category", "unknown"),
                "language": src["language"],
            })
            qid += 1
 
    pd.DataFrame(query_rows, columns=QUERY_COLUMNS).to_csv(
        TRAINING_QUERIES_CSV, mode="a", index=False,
        header=not os.path.exists(TRAINING_QUERIES_CSV),
    )
    query_rows = []
 
    elapsed = time.time() - t0
    done_songs = min((b + 1) * BATCH_SIZE, len(df_todo))
    print(f"batch {b+1}/{n_batches} done ({done_songs}/{len(df_todo)} target songs), "
          f"key #{current_index + 1}, {elapsed:.0f}s elapsed")
 
    time.sleep(RATE_LIMIT_SEC)
 
print("\nTargeted regeneration finished (or paused due to quota).")
 
# ------------------------------ SANITY CHECK -------------------------- #
 
df_q = pd.read_csv(TRAINING_QUERIES_CSV)
check = df_q[df_q["song_id"].isin(TARGET_SONG_IDS)]
print(f"\nQueries now present for target ids: {len(check)}")
print(check.groupby("song_id").size())

C:\Users\Sumit\AppData\Local\Temp\ipykernel_15380\4233452538.py:2: DtypeWarning: Columns (0: query_type) have mixed types. Specify dtype option on import or set low_memory=False.
  existing_q = pd.read_csv(TRAINING_QUERIES_CSV)


batch 1/1 done (1/1 target songs), key #1, 5s elapsed

Targeted regeneration finished (or paused due to quota).

Queries now present for target ids: 30
song_id
5643    30
dtype: int64


C:\Users\Sumit\AppData\Local\Temp\ipykernel_15380\4233452538.py:68: DtypeWarning: Columns (0: query_type) have mixed types. Specify dtype option on import or set low_memory=False.
  df_q = pd.read_csv(TRAINING_QUERIES_CSV)


In [32]:
# 1. Do these songs still have semantic profiles? (if not, that's the root cause)
df_sem = pd.read_csv("data/semantic_songs.csv")
check = df_sem[df_sem["song_id"].isin(range(5631, 5646))]
print(f"Profiles found: {len(check)}")
print(check[["song_id", "song_name"]])

Profiles found: 15
      song_id                        song_name
5630     5631                       dialogue 4
5631     5632     kahan chal diye idhar to aao
5632     5633                  chanda o chanda
5633     5634                           mittua
5634     5635                mama miya pom pom
5635     5636               humka maafi dai do
5636     5637  raahi naye naye rasta naya naya
5637     5638  sasu ji tune meri kadar na jani
5638     5639    hansne ki chah ne kitna mujhe
5639     5640   dhak dhak se dhadakna sikha de
5640     5641      jane kya dhoondti rahti hai
5641     5642         zindagi ka naam dostisad
5642     5643             main inpe marta hoon
5643     5644       pahla pahla pyar na bhoole
5644     5645   ek nazar dekh ke hum jaan gaye


In [33]:
EXPECTED_PER_SONG = len(CATEGORIES) * QUERIES_PER_CATEGORY  # 33

for item in result:
    n = len(item.get("queries", []))
    if n != EXPECTED_PER_SONG:
        print(f"  WARNING: song_id {item['song_id']} got {n} queries, expected {EXPECTED_PER_SONG}")

In [40]:

# if os.path.exists(TRAINING_QUERIES_CSV):
#     existing_q = pd.read_csv(TRAINING_QUERIES_CSV)
#     qid_start = int(existing_q["query_id"].max()) + 1 if len(existing_q) else 1
# qid = qid_start

# t0 = time.time()
# n_batches = (len(df_todo) + BATCH_SIZE - 1) // BATCH_SIZE

# for b in range(n_batches):
#     batch_df = df_todo.iloc[b * BATCH_SIZE: (b + 1) * BATCH_SIZE]
#     songs_batch = batch_df.to_dict("records")

#     result = call_llm(songs_batch)

#     if result == "ALL_KEYS_EXHAUSTED":
#         break
#     if result is None:
#         print(f"[batch {b}] failed on every key for non-quota reasons, skipping this batch.")
#         continue

#     for item in result:
#         sid = item["song_id"]
#         src_row = batch_df[batch_df["song_id"] == sid]
#         if src_row.empty:
#             continue
#         src = src_row.iloc[0]

#         for q in item.get("queries", []):
#             query_rows.append({
#                 "query_id": qid,
#                 "song_id": sid,
#                 "query": q.get("query", ""),
#                 "category": q.get("category", "unknown"),
#                 "generated_by": f"gemini:{GEMINI_MODEL}",
#                 "language": src["language"],
#             })
#             qid += 1

#     pd.DataFrame(query_rows, columns=QUERY_COLUMNS).to_csv(
#         TRAINING_QUERIES_CSV, mode="a", index=False,
#         header=not os.path.exists(TRAINING_QUERIES_CSV),
#     )
#     query_rows = []

#     elapsed = time.time() - t0
#     done_songs = min((b + 1) * BATCH_SIZE, len(df_todo))
#     eta_min = elapsed / done_songs * len(df_todo) / 60 if done_songs else 0
#     print(f"batch {b+1}/{n_batches} done ({done_songs}/{len(df_todo)} songs), "
#           f"key #{current_index + 1}, {elapsed:.0f}s elapsed, ~{eta_min:.1f} min ETA")

#     time.sleep(RATE_LIMIT_SEC)

# print("\nRun finished (or paused due to quota).")
# print(f"training_queries.csv -> {TRAINING_QUERIES_CSV}")

In [34]:
if os.path.exists(TRAINING_QUERIES_CSV):
    df_q = pd.read_csv(TRAINING_QUERIES_CSV)
    print(f"\nTotal queries: {len(df_q)}")
    print(f"Songs covered: {df_q['song_id'].nunique()}")
    print(f"Avg queries/song: {len(df_q) / max(df_q['song_id'].nunique(), 1):.2f}")
    print(df_q["category"].value_counts())


Total queries: 194779
Songs covered: 5902
Avg queries/song: 33.00
category
artist             17718
era                17718
emotion            17717
activity           17708
vocals             17707
language           17707
casual_hinglish    17706
similarity         17694
occasion           17661
mood               17661
conversational     17661
theme                 70
situation             45
genre                  6
Name: count, dtype: int64


C:\Users\Sumit\AppData\Local\Temp\ipykernel_15380\2429303097.py:2: DtypeWarning: Columns (0: query_type) have mixed types. Specify dtype option on import or set low_memory=False.
  df_q = pd.read_csv(TRAINING_QUERIES_CSV)


In [35]:
import pandas as pd

CSV_PATH = "data/training_queries.csv"

# 1. Load dataset
df = pd.read_csv(CSV_PATH, low_memory=False)

# 2. Sort so songs 5631-5645 are placed seamlessly between 5630 and 5646
df = df.sort_values(by=["song_id"]).reset_index(drop=True)

# 3. Renumber query_id sequentially from 1 to 194,779 (0 gaps)
df["query_id"] = range(1, len(df) + 1)

# 4. Save cleanly
df.to_csv(CSV_PATH, index=False)

print("Done! Dataset is now in perfect sequential order with zero gaps.")


Done! Dataset is now in perfect sequential order with zero gaps.
